# Graph-Aware Retrieval with Oracle AI Agent Memory

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oracle-devrel/oracle-ai-developer-hub/blob/main/notebooks/graph_aware_retrieval_agent_memory.ipynb) [![Docs](https://img.shields.io/badge/Docs-Oracle%20AI%20Agent%20Memory-red)](https://docs.oracle.com/en/database/oracle/agent-memory/)

This notebook shows how Graph Memory can help an agent reason over memory that changes over time. Instead of treating memories as an isolated flat list, Oracle AI Agent Memory can preserve typed relationships between memory records, such as a newer fact superseding an older one or one memory supporting another.

The use case is a customer-support workflow where an agent needs to remember evolving facts: an original delivery issue, a later preference update, a resolution, and a follow-up commitment. The notebook focuses on the release capabilities behind memory evolution: explicit record links, automatic linking during extraction, graph-aware retrieval, `num_hops`, `max_linked_results`, `include_invalid_results`, and `linked_results`.

The notebook is designed to run against FreeSQL, Autonomous AI Database, or Oracle AI Database connection details. It sets the required Developer Hub program identifier before creating the database connection and uses the managed schema policy to create or upgrade the Oracle AI Agent Memory schema.

## What This Notebook Demonstrates

This notebook focuses on the Graph Memory capabilities introduced for memory evolution:

- connecting Oracle AI Agent Memory through the release-relevant Oracle Database options: FreeSQL, Autonomous AI Database, or Oracle AI Database;
- initializing or upgrading the managed schema needed for graph-aware memory;
- creating durable support memories;
- linking records with typed relationships such as `supersedes` and `supports`;
- comparing direct search with graph-aware retrieval using `num_hops`;
- inspecting `linked_results` as prompt-ready relationship context;
- enabling automatic linking during memory extraction.

## Conceptual Flow: Memory as a Graph

This notebook uses one customer-support scenario to show how Oracle AI Agent Memory can treat memory as evolving context instead of a flat list of facts.

**Memory formation**

`Support conversation` -> `Durable memory records` -> `Typed memory links` -> `memory_graph`

**Graph-aware retrieval**

`User query` -> `Direct memory search` -> `Graph expansion` -> `Direct result + linked_results`

**Search behavior**

- `num_hops = 0`: returns direct memory matches.
- `num_hops = 1`: returns direct matches plus nearby linked memory context.

The key idea is simple: a newer support fact can supersede or support an older one, while the older memory remains available as historical context. Graph-aware retrieval gives the agent the current fact plus enough relationship context to understand how that fact evolved.

## Part 1 - Setup and Database Connection

Oracle AI Agent Memory stores and searches memory in Oracle AI Database. For this release notebook, the connection path is intentionally focused on hosted Oracle Database environments rather than local containers.

Start with **FreeSQL** when you want the fastest hosted demo setup. Use **Autonomous AI Database** when you are running against an ADB instance with wallet/client credentials. Use **Oracle AI Database** when your environment provides a standard service name, Easy Connect string, or full connect descriptor.

| Option | Why it is included | What you put in `.env` |
| --- | --- | --- |
| FreeSQL | Fastest path for a Developer Hub notebook because the hosted database and Python connection details are provided from the FreeSQL page. | `DB_USER`, `DB_PASSWORD`, `DB_DSN` |
| Autonomous AI Database | Best match for ADB-based validation of the release features, including managed schema creation or upgrade. | `DB_USER`, `DB_PASSWORD`, `DB_DSN`, optionally `DB_WALLET_LOCATION` |
| Oracle AI Database | General Oracle AI Database setup for teams that already have a service name, Easy Connect string, or connect descriptor. | `DB_USER`, `DB_PASSWORD`, `DB_DSN` |

### FreeSQL

FreeSQL is the easiest option for readers who need a hosted Oracle Database quickly. Open **Connect to the Database**, choose **Python**, then copy the username, generated password, and DSN into the notebook environment.

```env
DB_USER=<freesql-user>
DB_PASSWORD=<freesql-password>
DB_DSN=<freesql-python-dsn>
MEMORY_STORE_ID=graph_memory_release_demo
```

### Autonomous AI Database

Autonomous AI Database is the preferred ADB path for validating the release workflow against a managed Oracle Database instance. Use the database credentials and service name from the downloaded wallet/client credentials. If your runtime requires the wallet location explicitly, point `DB_WALLET_LOCATION` to the unzipped wallet directory.

```env
DB_USER=admin
DB_PASSWORD=<your-adb-password>
DB_DSN=<adb-service-name>
DB_WALLET_LOCATION=<path-to-unzipped-wallet>
MEMORY_STORE_ID=graph_memory_release_demo
```

### Oracle AI Database

For an existing Oracle AI Database environment, provide the database user, password, and DSN supplied by your database administrator or cloud setup. `DB_DSN` can be a service name, an Easy Connect string, or a full connect descriptor, depending on how the environment is configured.

```env
DB_USER=admin
DB_PASSWORD=<your-password>
DB_DSN=<your-service-name-or-connect-descriptor>
MEMORY_STORE_ID=graph_memory_release_demo
```

Before any database connection is created, the notebook sets the Developer Hub program identifier:

```python
oracledb.defaults.program = "devrel-developerhub-graph-aware-retrieval-agent-memory"
```

The same memory code runs after the connection pool is created, regardless of whether the credentials came from FreeSQL, Autonomous AI Database, or another Oracle AI Database setup.

### Install the Python Packages

Run this once in a clean Python environment. If the package was already imported in the current kernel, restart the kernel after installation and run the notebook again from the beginning.

In [ ]:
# Run this cell once in a fresh environment, then restart the kernel if packages were updated.
# If the required packages are already installed, you can skip this cell.
%pip install --upgrade "oracleagentmemory>=26.8" "oracledb>=2.5" pandas python-dotenv --quiet --disable-pip-version-check


### Configure the Runtime

Set the database and model provider values before running the notebook. For FreeSQL, open **Connect to the Database**, choose **Python**, and use the username, generated password, and DSN shown there. For Autonomous AI Database, use the service name from the wallet/client credentials and set the wallet location only when your runtime requires it.

Required values:

```text
DB_USER=<database user>
DB_PASSWORD=<database password>
DB_DSN=<database DSN, service name, or connect descriptor>
MODEL_PROVIDER_API_KEY=<model provider API key>
```

Optional values:

```text
DB_WALLET_LOCATION=<path-to-unzipped-adb-wallet>
MEMORY_LLM_MODEL=gpt-4o-mini
OAMP_MEMORY_STORE_ID=GRAPHMEMORYDEMO
```

The notebook reads these values from environment variables. A private `.env` file is also supported for convenience, but credentials are never printed.

### FreeSQL Connection Guide

Use this path when you want a hosted Oracle Database schema without setting up a local database.

<details>
<summary><strong>FreeSQL setup steps</strong></summary>

1. Sign in to [FreeSQL](https://freesql.com).
2. Open **Connect to the Database**.
3. Choose **Python**.
4. Copy the generated username, password, and DSN/connect descriptor.
5. Add them to your private `.env` file:

```env
DB_USER=<freesql-user>
DB_PASSWORD=<freesql-password>
DB_DSN=<freesql-python-dsn>
OAMP_MEMORY_STORE_ID=GRAPHMEMORYDEMO
```

</details>

**Setup flow**

`FreeSQL credentials` -> `python-oracledb connection` -> `table-permission check` -> `Agent Memory schema check` -> `run graph-aware retrieval`

The preflight below confirms that the database connection works before the notebook creates the Oracle AI Agent Memory store. For FreeSQL, the notebook can reuse an Agent Memory managed schema that is already available for the selected memory store ID. The preflight check makes that setup status visible before the graph workflow starts.


In [ ]:
import inspect
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import oracledb
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

oracledb.defaults.program = "devrel-developerhub-graph-aware-retrieval-agent-memory"

print("Runtime and Oracle Database program identifier: READY")


In [ ]:
import importlib.metadata as metadata


def parse_version_prefix(version):
    parts = []
    for part in version.split(".")[:2]:
        digits = "".join(ch for ch in part if ch.isdigit())
        parts.append(int(digits or 0))
    return tuple(parts)


package_version = metadata.version("oracleagentmemory")

print(f"oracleagentmemory package version: {package_version}")
if parse_version_prefix(package_version) < (26, 8):
    raise RuntimeError(
        "This notebook requires oracleagentmemory 26.8 or later. "
        "Run the install cell after the release package is available, restart the kernel, "
        "and then run the notebook again from the beginning."
    )

print("oracleagentmemory 26.8+ requirement: READY")


In [ ]:
def read_config():
    config = {
        "DB_USER": os.getenv("DB_USER") or os.getenv("ORACLE_USER"),
        "DB_PASSWORD": os.getenv("DB_PASSWORD") or os.getenv("ORACLE_PASSWORD"),
        "DB_DSN": os.getenv("DB_DSN") or os.getenv("ORACLE_DSN"),
        "MODEL_PROVIDER_API_KEY": os.getenv("MODEL_PROVIDER_API_KEY") or os.getenv("OPENAI_API_KEY"),
        "MEMORY_LLM_MODEL": os.getenv("MEMORY_LLM_MODEL", "gpt-4o-mini"),
        "MEMORY_STORE_ID": os.getenv("OAMP_MEMORY_STORE_ID", "GRAPHMEMORYDEMO"),
    }
    missing = [name for name, value in config.items() if name in {"DB_USER", "DB_PASSWORD", "DB_DSN", "MODEL_PROVIDER_API_KEY"} and not value]
    if missing:
        raise RuntimeError(
            "Missing required configuration values: " + ", ".join(missing)
        )
    return config

CONFIG = read_config()

print("Configuration: READY")

### Connect to Oracle AI Database

The preflight checks the connection and normal user-schema table operations without exposing schema names, DSNs, passwords, API keys, or local paths.

For FreeSQL, this notebook expects an Agent Memory managed schema to already be available for the selected memory store ID. The notebook verifies that setup before the graph workflow starts and reuses the existing schema when it is found.

For Autonomous AI Database or Oracle AI Database environments, the notebook can create or upgrade the managed schema when needed.


In [ ]:
db_pool = oracledb.create_pool(
    user=CONFIG["DB_USER"],
    password=CONFIG["DB_PASSWORD"],
    dsn=CONFIG["DB_DSN"],
    min=1,
    max=4,
    increment=1,
)


def is_freesql_connection(dsn):
    return "freesql" in str(dsn).lower()


IS_FREESQL = is_freesql_connection(CONFIG["DB_DSN"])


def run_preflight_check(name, fn, implication):
    try:
        detail = fn()
        return {
            "check": name,
            "status": "PASS",
            "detail": detail,
            "implication": implication["pass"],
        }
    except Exception as exc:
        return {
            "check": name,
            "status": "FAIL",
            "detail": f"{type(exc).__name__}: {exc}",
            "implication": implication["fail"],
        }


def fetch_rows(sql, **params):
    with db_pool.acquire() as connection:
        with connection.cursor() as cursor:
            cursor.execute(sql, params)
            return cursor.fetchall()


def check_connection():
    rows = fetch_rows("SELECT 1 FROM dual")
    if not rows:
        raise RuntimeError("SELECT 1 FROM dual returned no rows.")
    return "SELECT 1 FROM dual returned a row"


def check_table_operations():
    test_table = f"OAM_GRAPH_CHECK_{uuid4().hex[:8].upper()}"
    try:
        with db_pool.acquire() as connection:
            with connection.cursor() as cursor:
                cursor.execute(
                    f"""
                    CREATE TABLE {test_table} (
                        id NUMBER PRIMARY KEY,
                        note VARCHAR2(100)
                    )
                    """
                )
                cursor.execute(
                    f"INSERT INTO {test_table} (id, note) VALUES (:id, :note)",
                    id=1,
                    note="graph memory permission check",
                )
                row_count = cursor.execute(f"SELECT COUNT(*) FROM {test_table}").fetchone()[0]
                if row_count != 1:
                    raise RuntimeError("Table permission check returned an unexpected row count.")
            connection.commit()
        return "created, inserted, selected, and dropped a small table"
    finally:
        try:
            with db_pool.acquire() as cleanup_connection:
                with cleanup_connection.cursor() as cursor:
                    cursor.execute(f"DROP TABLE {test_table} PURGE")
                cleanup_connection.commit()
        except oracledb.Error:
            pass


def managed_schema_exists(memory_store_id):
    prefix = memory_store_id.upper()
    rows = fetch_rows(
        """
        SELECT table_name
        FROM user_tables
        WHERE table_name LIKE :prefix
        FETCH FIRST 1 ROWS ONLY
        """,
        prefix=f"{prefix}_%",
    )
    return bool(rows)


MANAGED_SCHEMA_EXISTS = managed_schema_exists(CONFIG["MEMORY_STORE_ID"])

preflight_rows = [
    run_preflight_check(
        "connection",
        check_connection,
        {
            "pass": "python-oracledb can connect to the configured database.",
            "fail": "Fix the database connection values before continuing.",
        },
    ),
    run_preflight_check(
        "table_operations",
        check_table_operations,
        {
            "pass": "The schema can create and manage normal user tables.",
            "fail": "The schema cannot run the basic table operations needed by the notebook.",
        },
    ),
    {
        "check": "managed_schema_exists",
        "status": "PASS" if MANAGED_SCHEMA_EXISTS else "MISSING",
        "detail": f"memory_store_id={CONFIG['MEMORY_STORE_ID']}",
        "implication": "The notebook can reuse the existing Agent Memory managed schema."
        if MANAGED_SCHEMA_EXISTS
        else "The Agent Memory managed schema was not found for this memory store ID.",
    },
    {
        "check": "database_path",
        "status": "FREESQL" if IS_FREESQL else "FULL_DATABASE",
        "detail": "FreeSQL connection detected" if IS_FREESQL else "ADB or Oracle AI Database connection detected",
        "implication": "FreeSQL uses an existing managed schema for this notebook."
        if IS_FREESQL
        else "This database path can create or upgrade the managed schema when needed.",
    },
]

preflight = pd.DataFrame(preflight_rows)
display(preflight)

required = preflight[preflight["check"].isin(["connection", "table_operations"])]
if not (required["status"] == "PASS").all():
    raise RuntimeError("Database preflight failed. Fix the connection/table checks before continuing.")

if IS_FREESQL and not MANAGED_SCHEMA_EXISTS:
    raise RuntimeError(
        "FreeSQL connection works, but the Agent Memory managed schema was not found "
        "for this memory_store_id. For FreeSQL, use the default/pre-created memory "
        "store ID provided for Developer Hub notebooks, then rerun from this cell."
    )

if MANAGED_SCHEMA_EXISTS:
    print("Agent Memory managed schema: FOUND")
else:
    print("Agent Memory managed schema was not found. This database path will create or upgrade it in the next setup step.")

print("Oracle AI Database connection: READY")
print("FreeSQL preflight: COMPLETE" if IS_FREESQL else "Database preflight: COMPLETE")


## Part 2 - Build the Memory Client

This section creates the model interfaces and database-backed memory store. The store uses the package-managed schema so that memory records, metadata, search indexes, and memory graph structures are owned by Oracle AI Agent Memory.

In [ ]:
from oracleagentmemory.core import OracleAgentMemory, OracleDBMemoryStore, SchemaPolicy

try:
    from oracleagentmemory.core import MemoryExtractionConfig
except ImportError as exc:
    raise RuntimeError("This notebook requires an oracleagentmemory package with MemoryExtractionConfig support.") from exc

print("Oracle AI Agent Memory imports: READY")

In [ ]:
from oracleagentmemory.core import SearchStrategy
from oracleagentmemory.core.embedders import Embedder
from oracleagentmemory.core.llms import Llm, LlmApiType

llm_kwargs = {
    "model": CONFIG["MEMORY_LLM_MODEL"],
    "api_key": CONFIG["MODEL_PROVIDER_API_KEY"],
    "temperature": 0,
    "max_tokens": 2_000,
}
if os.getenv("MEMORY_LLM_API_BASE"):
    llm_kwargs["api_base"] = os.getenv("MEMORY_LLM_API_BASE")
if os.getenv("MEMORY_LLM_API_TYPE", "chat_completions") == "responses":
    llm_kwargs["api_type"] = LlmApiType.RESPONSES

memory_llm = Llm(**llm_kwargs)

embedder = Embedder(
    model=os.getenv("MEMORY_EMBEDDING_MODEL") or os.getenv("EMBED_MODEL") or "text-embedding-3-small",
    api_key=CONFIG["MODEL_PROVIDER_API_KEY"],
    embedding_dimension=int(os.getenv("MEMORY_EMBEDDING_DIMENSION") or os.getenv("EMBED_DIM") or "1536"),
    max_input_tokens=512,
    normalize=True,
)

SEARCH_STRATEGY = SearchStrategy.VECTOR
VECTOR_DIM = int(os.getenv("MEMORY_EMBEDDING_DIMENSION") or os.getenv("EMBED_DIM") or "1536")

print("Memory extraction LLM: READY")
print("Retrieval route: provider embeddings with Oracle AI Database vector search")

In [ ]:
schema_policy = SchemaPolicy.REQUIRE_EXISTING if MANAGED_SCHEMA_EXISTS else SchemaPolicy.CREATE_IF_NECESSARY

store_kwargs = {
    "pool": db_pool,
    "embedder": embedder,
    "memory_store_id": CONFIG["MEMORY_STORE_ID"],
    "schema_policy": schema_policy,
    "search_strategy": SEARCH_STRATEGY,
}
if VECTOR_DIM is not None:
    store_kwargs["vector_dim"] = VECTOR_DIM

try:
    store = OracleDBMemoryStore(**store_kwargs)
except Exception as exc:
    if IS_FREESQL:
        raise RuntimeError(
            "FreeSQL connection works, but the notebook could not initialize the "
            "Agent Memory managed schema for this memory_store_id. Confirm that "
            "the selected FreeSQL memory store ID points to a pre-created Developer "
            "Hub managed schema, then rerun from the preflight cell."
        ) from exc
    raise

base_extraction_config = MemoryExtractionConfig(
    memory_extraction_frequency=1,
    enable_context_summary=False,
)

memory = OracleAgentMemory(
    store=store,
    llm=memory_llm,
    memory_extraction_config=base_extraction_config,
)

print("Database-backed graph memory client: READY")
print("Managed schema policy:", schema_policy)
print("Managed schema note: Oracle AI Agent Memory 26.8 uses schema version 13 for memory links and graph-aware retrieval.")


### Verify Graph Memory API Availability

Graph-aware retrieval requires a package release that supports memory links and linked search results. This cell checks the public methods and arguments before the workflow starts.

In [ ]:
def has_parameter(callable_obj, parameter_name):
    try:
        return parameter_name in inspect.signature(callable_obj).parameters
    except (TypeError, ValueError):
        return False

api_checks = pd.DataFrame(
    [
        {"Capability": "Create explicit record links", "Check": "OracleAgentMemory.link_records", "Available": hasattr(memory, "link_records")},
        {"Capability": "Update record links", "Check": "OracleAgentMemory.update_record_link", "Available": hasattr(memory, "update_record_link")},
        {"Capability": "Delete record links", "Check": "OracleAgentMemory.delete_record_link", "Available": hasattr(memory, "delete_record_link")},
        {"Capability": "Search with graph hops", "Check": "search(..., num_hops=...)", "Available": has_parameter(memory.search, "num_hops") if hasattr(memory, "search") else False},
        {"Capability": "Limit linked results", "Check": "search(..., max_linked_results=...)", "Available": has_parameter(memory.search, "max_linked_results") if hasattr(memory, "search") else False},
        {"Capability": "Control invalid top-level results", "Check": "search(..., include_invalid_results=...)", "Available": has_parameter(memory.search, "include_invalid_results") if hasattr(memory, "search") else False},
    ]
)

display(api_checks)

if not api_checks["Available"].all():
    missing = ", ".join(api_checks.loc[~api_checks["Available"], "Capability"])
    raise RuntimeError(
        "Install oracleagentmemory 26.8 or later. Missing graph memory capabilities: " + missing
    )

print("Graph memory APIs: READY")

## Part 3 - Memory Evolution Use Case

The workflow uses support memories that evolve over time:

- a customer initially prefers morning delivery windows;
- a later conversation changes the preference to afternoon delivery;
- a replacement shipment update supports the new preference;
- an older memory is retained as historical context rather than deleted.

This is the kind of situation where graph-aware retrieval is useful: the agent needs the latest usable fact, but the relationship to older facts still matters.

In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
USER_ID = f"customer_graph_{RUN_ID}"
AGENT_ID = "support-agent"
THREAD_ID = f"graph_memory_support_{RUN_ID}"

seed_memories = [
    {
        "label": "original_preference",
        "record_type": "preference",
        "text": "Customer prefers morning delivery windows for replacement shipments.",
        "metadata": {"tenant": "demo", "case_id": "CASE-8421", "topic": "delivery_preference", "state": "historical"},
    },
    {
        "label": "updated_preference",
        "record_type": "preference",
        "text": "Customer now prefers afternoon delivery windows for replacement shipments because mornings conflict with work.",
        "metadata": {"tenant": "demo", "case_id": "CASE-8421", "topic": "delivery_preference", "state": "current"},
    },
    {
        "label": "replacement_context",
        "record_type": "fact",
        "text": "Replacement shipment RMA-8842 is tied to order ORD-7421 and should use the current delivery-window preference.",
        "metadata": {"tenant": "demo", "case_id": "CASE-8421", "topic": "replacement", "state": "current"},
    },
]

pd.DataFrame(seed_memories)

## Part 4 - Baseline and Explicit Links

First, run a flat-memory baseline before any record links are created. This shows the limitation of independent memory records: direct search can retrieve relevant facts, but it does not explain which memory supersedes another or which record supports the current state.

After the baseline, the notebook creates typed links so the same facts can be retrieved as connected memory.


In [ ]:
created = {}

for item in seed_memories:
    result = memory.add_memory(
        user_id=USER_ID,
        agent_id=AGENT_ID,
        text=item["text"],
        metadata=item["metadata"],
    )
    created[item["label"]] = result

memory_rows = []
for label, result in created.items():
    memory_rows.append(
        {
            "label": label,
            "memory_id": getattr(result, "memory_id", getattr(result, "id", None)),
            "text": getattr(result, "text", seed_memories[[m["label"] for m in seed_memories].index(label)]["text"]),
        }
    )

memory_table = pd.DataFrame(memory_rows)
display(memory_table)
print("Seed memories: READY")

### Baseline: Search Before Graph Links

Before creating `supersedes` or `supports` links, the memories are independent records. A direct search can return the original preference, updated preference, or replacement context, but it cannot expose the relationship between them through `linked_results`.


In [ ]:
def preview_result_text(result):
    return getattr(result, "text", getattr(result, "memory", getattr(result, "content", "")))

baseline_query = "What delivery window should I use for replacement shipment RMA-8842?"
baseline_results = memory.search(
    query=baseline_query,
    user_id=USER_ID,
    agent_id=AGENT_ID,
    num_hops=0,
    max_linked_results=0,
    include_invalid_results=True,
)

baseline_rows = []
for rank, result in enumerate(baseline_results, start=1):
    baseline_rows.append(
        {
            "rank": rank,
            "memory": preview_result_text(result),
            "linked_result_count": len(getattr(result, "linked_results", []) or []),
        }
    )

display(pd.DataFrame(baseline_rows))
print("Flat-memory baseline: READY")


### What the Baseline Shows

The baseline is useful because it shows what direct search can and cannot do. It can retrieve relevant text, but without graph links it does not identify that the updated delivery preference supersedes the original preference, or that the replacement context supports the current preference. The next cell adds those relationships explicitly.


In [ ]:
def memory_id(label):
    result = created[label]
    value = getattr(result, "memory_id", getattr(result, "id", None))
    if value is None:
        raise RuntimeError(f"Could not read memory id for {label} from add_memory result.")
    return value

def record_type(label):
    for item in seed_memories:
        if item["label"] == label:
            return item["record_type"]
    raise KeyError(label)

links_to_create = [
    {
        "source_label": "updated_preference",
        "target_label": "original_preference",
        "relation_type": "supersedes",
        "reason": "The customer's delivery-window preference changed from morning to afternoon.",
    },
    {
        "source_label": "replacement_context",
        "target_label": "updated_preference",
        "relation_type": "supports",
        "reason": "The replacement shipment should follow the current delivery-window preference.",
    },
]

created_links = []
for link in links_to_create:
    created_link = memory.link_records(
        source_record_id=memory_id(link["source_label"]),
        source_record_type=record_type(link["source_label"]),
        target_record_id=memory_id(link["target_label"]),
        target_record_type=record_type(link["target_label"]),
        relation_type=link["relation_type"],
        metadata={"reason": link["reason"], "created_by": "notebook"},
    )
    created_links.append(created_link)

link_table = pd.DataFrame(
    [
        {
            "relationship": item["relation_type"],
            "from_memory": item["source_label"],
            "to_memory": item["target_label"],
            "why": item["reason"],
        }
        for item in links_to_create
    ]
)

display(link_table)
print("Explicit record links: READY")

## Part 5 - Compare Direct Search with Graph-Aware Search

Now run the same question after the graph links exist. This is the before-and-after comparison: direct search still returns direct matches, while graph-aware search can include linked context that explains memory evolution.

Use `num_hops=0` for direct retrieval and `num_hops=1` when the response benefits from one step of linked memory context. Use `max_linked_results` to bound how much linked context can be returned with each direct result.

`include_invalid_results` controls whether invalid memories can appear as top-level direct results. Historical or invalid memories may still appear as linked context or traversal intermediates, because they can explain how a valid memory evolved.


### Inspect the Search Results

The search output is organized so the graph behavior is easy to inspect:

- the direct memory record returned by the query;
- the linked memory records that explain the surrounding state;
- the relationship type, such as `supersedes`, `supports`, or `refines`.

The exact wording of extracted memories can vary because extraction is model-assisted. The retrieval pattern is the important part: graph-aware search returns connected context that direct search would not include by itself.

In [ ]:
def search_memories(query, num_hops=0, max_linked_results=3, include_invalid_results=True):
    return memory.search(
        query=query,
        user_id=USER_ID,
        agent_id=AGENT_ID,
        num_hops=num_hops,
        max_linked_results=max_linked_results,
        include_invalid_results=include_invalid_results,
    )

query = "What delivery window should I use for replacement shipment RMA-8842?"
flat_results = search_memories(query, num_hops=0, include_invalid_results=False)
graph_results = search_memories(query, num_hops=1, max_linked_results=5, include_invalid_results=True)

print("Direct search: READY")
print("Graph-aware search: READY")

In [ ]:
def result_text(result):
    return getattr(result, "text", getattr(result, "memory", getattr(result, "content", "")))

def result_score(result):
    return getattr(result, "score", getattr(result, "relevance_score", None))

def linked_results(result):
    return getattr(result, "linked_results", []) or []

def summarize_results(results, label):
    rows = []
    for rank, result in enumerate(results, start=1):
        rows.append(
            {
                "search_mode": label,
                "rank": rank,
                "score": result_score(result),
                "memory": result_text(result),
                "linked_result_count": len(linked_results(result)),
            }
        )
    return pd.DataFrame(rows)

comparison = pd.concat(
    [
        summarize_results(flat_results, "num_hops=0"),
        summarize_results(graph_results, "num_hops=1"),
    ],
    ignore_index=True,
)

display(comparison)

In [ ]:
linked_rows = []
for parent_rank, result in enumerate(graph_results, start=1):
    for linked_rank, linked in enumerate(linked_results(result), start=1):
        linked_rows.append(
            {
                "parent_rank": parent_rank,
                "linked_rank": linked_rank,
                "linked_memory": result_text(linked),
                "relationship": getattr(linked, "link_type", getattr(linked, "relationship", "linked")),
            }
        )

if linked_rows:
    display(pd.DataFrame(linked_rows))
else:
    print("No linked results returned for this query. Try increasing num_hops or max_linked_results.")

## Part 6 - Automatic Linking During Extraction

Explicit links are useful when an application already knows the relationship. Automatic linking is useful when the extraction workflow identifies that a newly extracted memory relates to existing memories during or after extraction.

`MemoryExtractionConfig.memory_link_extraction_mode` controls this behavior through the public `MemoryLinkExtractionMode` enum:

- `POST_EXTRACTION` is the default. The SDK writes extracted memories first, then searches candidate memories and persists validated links as a best-effort follow-up operation.
- `DURING_EXTRACTION` presents candidate memories during extraction so the LLM can select a relationship while forming the memory.
- `DISABLED` keeps extraction behavior without automatic memory linking.

`add_memory()` also exposes `autonomous_linking`, which lets directly added memories participate in automatic link discovery when the memory client is configured with an LLM.

In [ ]:
def enum_value(enum_class, preferred_names):
    for name in preferred_names:
        if hasattr(enum_class, name):
            return getattr(enum_class, name)
    return None

try:
    from oracleagentmemory.core import MemoryLinkExtractionMode
except Exception:
    MemoryLinkExtractionMode = None

link_mode = None
if MemoryLinkExtractionMode is not None:
    link_mode = enum_value(MemoryLinkExtractionMode, ["POST_EXTRACTION"])

extraction_kwargs = {
    "memory_extraction_frequency": 1,
    "enable_context_summary": False,
}
if link_mode is not None:
    extraction_kwargs["memory_link_extraction_mode"] = link_mode

auto_link_config = MemoryExtractionConfig(**extraction_kwargs)

auto_memory = OracleAgentMemory(
    store=store,
    llm=memory_llm,
    memory_extraction_config=auto_link_config,
)

print("Automatic linking extraction config: READY")
print("Selected mode:", link_mode)

In [ ]:
auto_thread = auto_memory.create_thread(
    thread_id=f"auto_graph_support_{RUN_ID}",
    user_id=USER_ID,
    agent_id=AGENT_ID,
)

await auto_thread.add_messages_async(
    [
        {
            "role": "user",
            "content": "For order ORD-7421, please stop using morning delivery. Afternoon is now the right delivery window.",
        },
        {
            "role": "assistant",
            "content": "I will remember that order ORD-7421 should use afternoon delivery going forward.",
        },
        {
            "role": "user",
            "content": "This is for replacement shipment RMA-8842, tied to the same case.",
        },
    ],
    metadata={"tenant": "demo", "case_id": "CASE-8421", "source": "support_chat"},
)

await auto_thread.wait_for_memory_extraction_async()

print("Automatic-linking extraction thread: READY")

In [ ]:
if has_parameter(memory.add_memory, "autonomous_linking"):
    auto_added = memory.add_memory(
        user_id=USER_ID,
        agent_id=AGENT_ID,
        text="Customer asked that future replacement shipments avoid early morning delivery windows.",
        metadata={"tenant": "demo", "case_id": "CASE-8421", "topic": "delivery_preference", "state": "current"},
        autonomous_linking=True,
    )
    print("Direct add_memory autonomous_linking example: READY")
else:
    print("autonomous_linking is not available in this installed package version; continuing with the extraction-based linking example.")

auto_results = search_memories(
    "What is the current delivery preference for ORD-7421?",
    num_hops=1,
    max_linked_results=5,
    include_invalid_results=True,
)

auto_table = summarize_results(auto_results, "automatic linking search")
display(auto_table)

print("Automatic-linking retrieval: READY")

## Part 7 - Render Prompt-Ready Graph Context

When search results include linked memories, an agent does not need to replay the full conversation. It can inject a compact view of the direct match plus the related memory context.

In [ ]:
if graph_results:
    top_result = graph_results[0]
    if hasattr(top_result, "format_content"):
        print(top_result.format_content())
    else:
        print(result_text(top_result))
        for linked in linked_results(top_result):
            print("- linked:", result_text(linked))
else:
    print("No graph-aware results to render.")

## Output Review Checklist

Use the final output to confirm the graph-aware retrieval behavior:

- the current delivery preference appears as the direct answer;
- the older preference remains available as linked historical context;
- replacement shipment context supports the current preference;
- graph-aware search exposes related memories through `linked_results`;
- direct search with `num_hops=0` stays simple when graph context is not needed;
- `include_invalid_results=False` excludes invalid memories from the top-level direct result list, while linked historical context can still explain how the current memory evolved.

## Cleanup and Shutdown

The cleanup cell closes the database pool. If you want to inspect the created package-managed records after the run, keep the records in place and only close the pool.

In [ ]:
try:
    db_pool.close(force=True)
except Exception:
    pass

print("Cleanup and shutdown: READY")

## Summary

Graph-aware retrieval helps Oracle AI Agent Memory represent memory as evolving context rather than a flat list of isolated facts. Applications can create explicit record links when relationships are known, and use automatic linking during extraction when the configured memory workflow identifies relationships from conversation history.

For agent developers, the practical pattern is:

1. store durable memories in Oracle AI Database;
2. initialize or upgrade the managed schema so schema version 13 graph-memory objects are available;
3. link records when facts supersede, refine, support, duplicate, or contradict each other;
4. use `num_hops`, `max_linked_results`, and `include_invalid_results` to control how much graph context search returns;
5. pass compact graph-aware results into the agent prompt only when that context helps the next response.